# What is class imbalance, and how do you handle it?

Class imbalance means one outcome is much rarer than the other, like fraud among normal payments. It’s the normal case in real problems, and it changes how you measure and train.

From the Hello Model glossary: https://sayhellomodel.com/glossary/class-imbalance/

## Why it’s a problem

Most interesting things are rare: fraud, defects, machine failures, customers about to leave, a disease in a screening test. When 1 transaction in 500 is fraud, a model that always says “not fraud” is 99.8% accurate and catches nothing. Two things go wrong:

- **Accuracy stops meaning anything,** so you can’t tell a useful model from a useless one.
- **The model learns to ignore the rare class,** because always guessing the common one is an easy way to be right most of the time.

## Step 1: Measure it properly

Before changing anything about training, change the metric. Use precision and recall on the rare class, the F1 score, or average precision. Always compare against a baseline that predicts the common class, so you can see whether the model adds anything.

## Step 2: Choose the threshold on purpose

This is often the single most effective fix. A model trained on imbalanced data gives rare cases low scores, so the default threshold of 0.5 flags almost nothing. Lower the threshold until precision and recall match what your problem needs, using a validation set, never the test set. Better still, pick the threshold with the lowest total cost of missed cases and false alarms.

## Step 3: Weight the rare class

Most libraries can make mistakes on the rare class count more during training. In scikit-learn, set `class_weight="balanced"`; in LightGBM or XGBoost, use `scale_pos_weight`. This often helps simple models such as logistic regression. Gradient-boosted trees usually do well without it, as long as you choose the threshold carefully.

**Class weights in scikit-learn**

In [ ]:
from sklearn.linear_model import LogisticRegression

# Mistakes on the rare class count more, in proportion to how rare it is
model = LogisticRegression(class_weight="balanced", max_iter=1000)

## What about oversampling?

Copying rare examples or generating synthetic ones (such as SMOTE) is popular, but it rarely beats class weights plus a well-chosen threshold, and it’s easy to get wrong. If you resample, do it only on the training data, after the split; resampling before the split puts copies of the same case in both sets, which is data leakage.

## Get more of the rare class

The best fix is usually more real examples of the rare outcome: a longer history, labels from another source, or reviewing cases the model is unsure about. A few hundred more real fraud cases help more than any resampling trick.

> **Tip:** If the rare class is very rare and its cases are varied or new, consider anomaly detection instead: learn what normal looks like and flag what isn’t.